# 📘 Notebook 08: When the Table Is Too Big

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module C: Deep Value-Based Learning · Notebook 1 of 4 in this module · (8 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why a Q-table cannot be used when the state is made of **real numbers**
- Turn a continuous state into table rows by **discretisation**, and explain the trade-off in choosing the number of bins
- Describe **function approximation**: a model with parameters that outputs action values
- Build a small neural network that maps a state to **one value per action**
- Write the Q-learning update as a **loss function** and train the network with it
- Show with measurements that this naive combination is **unstable**, and name the two causes

> **Prerequisites:** Notebooks 05 and 07 (Q-learning with a table). From *From Python to Deep Learning & Fine-Tuning*: Notebook 12 (what a neural network is) and Notebook 13 (PyTorch, loss functions, the training loop, optimisers). This notebook does not re-teach those.
>
> 🔭 **Why this notebook matters later:** every remaining agent in the course is a neural network. This notebook makes the step from table to network as small as possible, and is honest about what breaks. Notebook 09 repairs it.

> 📦 **Libraries used in this notebook.**
>
> - **PyTorch** (installed as `torch`) provides the neural network and its training. It is already present in Google Colab.
> - **Gymnasium** provides CartPole, and **imageio** the videos.
>
> No GPU is needed: the networks are tiny. The longer training cells take a minute or two each.

In [ ]:
%pip install -q "gymnasium[classic-control]>=1.3" imageio imageio-ffmpeg torch

In [ ]:
import base64
import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
import torch
import torch.nn as nn
from IPython.display import HTML, display

torch.set_num_threads(2)


def show_video(frames, fps=30, width=420):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. A state made of real numbers

We met **CartPole** in Notebook 02. A pole is hinged on a cart, the agent pushes the cart left (action 0) or right (action 1), and it receives +1 for every step on which the pole has not fallen. An episode ends when the pole tips more than 12 degrees, when the cart leaves the track, or after 500 steps. So 500 is a perfect score.

The state is four real numbers:

| Index | Meaning | Typical range |
|---|---|---|
| 0 | position of the cart | −2.4 to 2.4 |
| 1 | speed of the cart | about −3 to 3 |
| 2 | angle of the pole (radians) | −0.21 to 0.21 |
| 3 | how fast the pole is turning | about −3 to 3 |

In [ ]:
env = gym.make("CartPole-v1")
state, info = env.reset(seed=0)
print("a state:", state)

states = set()
for episode in range(20):
    state, info = env.reset(seed=episode)
    terminated = truncated = False
    while not (terminated or truncated):
        states.add(tuple(state))
        state, reward, terminated, truncated, info = env.step(env.action_space.sample())
print(f"20 random episodes visited {len(states)} states, all different")

In Snake the agent met the same 256 observations again and again, and that repetition is what filled the table. Here **no state ever occurs twice**. A table with one row per state would have every row visited once at most. The agent has to carry over what it learns in one state to the states **near** it.

## 2. First idea: cut the numbers into bins

### Intuition first
A thermometer shows 21.37 degrees, and we say "about 21". Rounding turns infinitely many values into a few, and treats nearby values as the same. If we cut each of the four numbers into a few intervals, called **bins**, every state falls into one combination of bins, and a combination can be a row of a table.

### Formal definition
**Discretisation** replaces each continuous variable by the index of the interval it lies in. With $b$ bins for each of $d$ variables there are $b^d$ rows.

In [ ]:
LOW = np.array([-2.4, -3.0, -0.21, -3.0])
HIGH = -LOW


def make_binner(bins):
    """Returns a function that turns a state into four bin numbers."""
    edges = [np.linspace(LOW[i], HIGH[i], bins + 1)[1:-1] for i in range(4)]
    return lambda state: tuple(int(np.digitize(state[i], edges[i])) for i in range(4))


to_bins = make_binner(10)
state, info = env.reset(seed=0)
print("state:", np.round(state, 3), "-> bins", to_bins(state))

With that, Q-learning is the algorithm of Notebook 05, unchanged. The table simply has four index dimensions, one per variable.

In [ ]:
def binned_q_learning(env, bins, episodes=2000, alpha=0.1, gamma=0.99, seed=0):
    rng = np.random.default_rng(seed)
    to_bins = make_binner(bins)
    Q = np.zeros((bins, bins, bins, bins, 2))
    for episode in range(episodes):
        epsilon = max(0.01, 1 - episode / (0.5 * episodes))
        state, info = env.reset(seed=seed * 100000 + episode)
        s = to_bins(state)
        terminated = truncated = False
        while not (terminated or truncated):
            action = int(rng.integers(2)) if rng.random() < epsilon else int(Q[s].argmax())
            state, reward, terminated, truncated, info = env.step(action)
            s2 = to_bins(state)
            target = reward + gamma * (0.0 if terminated else Q[s2].max())
            Q[s + (action,)] += alpha * (target - Q[s + (action,)])
            s = s2
    return Q, to_bins


def evaluate(env, policy, episodes=30):
    """Average length of an episode when the policy is followed without exploration."""
    total = 0.0
    for episode in range(episodes):
        state, info = env.reset(seed=90000 + episode)
        terminated = truncated = False
        while not (terminated or truncated):
            state, reward, terminated, truncated, info = env.step(policy(state))
            total += reward
    return total / episodes


random_score = evaluate(env, lambda state: env.action_space.sample())
print(f"random policy: the pole stays up for {random_score:.0f} steps on average\n")

tables = {}
for bins in (3, 10, 20):
    Q, to_bins = binned_q_learning(env, bins)
    tables[bins] = (Q, to_bins)
    score = evaluate(env, lambda state: int(Q[to_bins(state)].argmax()))
    visited = int((Q != 0).any(axis=-1).sum())
    print(f"{bins:>2} bins per variable: {bins ** 4:>7,} rows, {visited:>5,} of them visited, "
          f"average episode length {score:5.0f}")

All three tables are far better than random play, so the idea works. Look at how the result depends on the number of bins:

- With **3 bins** the table is tiny and learning is quick, but the description is too coarse. A pole leaning slightly and a pole about to fall land in the same bin, and the agent cannot treat them differently.
- With **20 bins** the description is fine enough, but there are 160,000 rows. Each one is visited rarely, and what is learned in one bin tells the agent nothing about the bin next to it. After the same number of games it has learned less.
- **10 bins** sits between the two.

### Why this does not scale
The number of rows is $b^d$. Four variables with 10 bins give 10,000 rows. A robot arm with 12 measured quantities would need $10^{12}$. A small greyscale picture of 84 × 84 pixels, the input used for video games in Notebook 11, has 7,056 variables. This growth is called the **curse of dimensionality**, and no amount of computing power gets round it.

Discretisation also leaves the choice of bins to us, as the choice of features was left to us in Snake. We want a method that decides for itself which differences between states matter.

## 3. Second idea: a function with parameters

### Intuition first
A table stores one independent number for every state and action. A formula is different: the handful of numbers in $y = 2x + 1$ describe the value at **every** $x$, including values of $x$ nobody has tried. If we could describe the action values by a formula with adjustable numbers in it, a limited amount of experience could fix those numbers, and the formula would then give an answer everywhere.

### Formal definition
In **function approximation** the action values are computed by a function $\hat{q}(s, a; \mathbf{w})$ with **parameters** (or weights) $\mathbf{w}$. Learning means adjusting $\mathbf{w}$. The function we use is a **neural network**:

- **input**: the state (4 numbers for CartPole),
- **output**: one number per action (2 numbers), the estimated $q$-value of each.

One pass through the network gives the values of all actions at once, so choosing the best action is still an `argmax`.

In [ ]:
def make_network(hidden=64):
    return nn.Sequential(
        nn.Linear(4, hidden), nn.ReLU(),
        nn.Linear(hidden, hidden), nn.ReLU(),
        nn.Linear(hidden, 2),                 # one output per action
    )


torch.manual_seed(0)
network = make_network()
state, info = env.reset(seed=0)

with torch.no_grad():
    values = network(torch.as_tensor(state))

print("state:           ", np.round(state, 3))
print("action values:   ", values.numpy().round(3), "(untrained, so meaningless)")
print("greedy action:   ", int(values.argmax()))
print("parameters:      ", sum(p.numel() for p in network.parameters()))

The network has 4,610 parameters. The 20-bin table had 320,000 entries and could not tell two states in the same bin apart. The network gives a different answer for every state, with seventy times fewer numbers.

### Why it matters: generalisation
In a table, an update changes one entry and nothing else. In a network, the parameters are shared by all states, so an update for one state moves the values of **other states too**. That is the property we wanted: experience in one state informs its neighbours. Section 6 will show that it is also the source of trouble.

## 4. Q-learning as a loss function

### Formal definition
The tabular update moved $Q(S, A)$ a step towards the target $R + \gamma \max_a Q(S', a)$. A network has no entry to move, but it can be trained to **reduce the error** between its output and the same target. For one step $(S, A, R, S')$:

$$\text{target} = R + \gamma \max_{a} \hat{q}(S', a; \mathbf{w}) \qquad\qquad \text{loss} = \big(\text{target} - \hat{q}(S, A; \mathbf{w})\big)^2$$

The loss is the squared **TD error** of Notebook 05. One step of gradient descent on it adjusts the weights so that $\hat{q}(S, A)$ moves towards the target.

There is one subtle point. The target is computed by the same network, so it depends on $\mathbf{w}$ as well. We treat it as a **fixed number** and do not let gradients flow through it (`torch.no_grad()` below). Otherwise the network could reduce the loss by changing the target to match its guess, which is the wrong way round.

The function below is the most direct translation of Notebook 05 into PyTorch: act, observe one step, update on that one step, forget it.

In [ ]:
def naive_q_learning(env, steps=20000, learning_rate=1e-3, gamma=0.99, hidden=64, seed=0):
    """Q-learning with a network, updated on each step as it happens. Returns the network and the episode lengths."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    network = make_network(hidden)
    optimiser = torch.optim.Adam(network.parameters(), lr=learning_rate)
    lengths, length = [], 0
    state, info = env.reset(seed=seed)
    for step in range(steps):
        epsilon = max(0.01, 1 - step / (0.5 * steps))
        s = torch.as_tensor(state)
        if rng.random() < epsilon:
            action = int(rng.integers(2))
        else:
            with torch.no_grad():
                action = int(network(s).argmax())
        state, reward, terminated, truncated, info = env.step(action)
        length += 1

        with torch.no_grad():                                   # the target is treated as a fixed number
            future = 0.0 if terminated else network(torch.as_tensor(state)).max()
            target = reward + gamma * future
        loss = (target - network(s)[action]) ** 2               # squared TD error
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        if terminated or truncated:
            lengths.append(length)
            length = 0
            state, info = env.reset()
    return network, np.array(lengths)

## 5. What happens

We train three agents that differ only in their random seed, for 20,000 steps each. This cell takes about a minute.

In [ ]:
runs = [naive_q_learning(env, seed=seed) for seed in range(3)]


def smooth(values, window=10):
    return np.convolve(values, np.ones(window) / window, mode="valid")


plt.figure(figsize=(8, 4))
for seed, (network, lengths) in enumerate(runs):
    plt.plot(smooth(lengths), label=f"seed {seed}")
plt.axhline(500, color="gray", linestyle=":", label="perfect score")
plt.title("Naive Q-learning with a network: episode length during training")
plt.xlabel("episode"); plt.ylabel("steps the pole stayed up (smoothed)"); plt.grid(True); plt.legend()
plt.show()

for seed, (network, lengths) in enumerate(runs):
    best = smooth(lengths).max()
    greedy = evaluate(env, lambda state: int(network(torch.as_tensor(state)).argmax()))
    print(f"seed {seed}: best average over 10 training episodes {best:4.0f},  "
          f"last 10 training episodes {lengths[-10:].mean():4.0f},  final policy evaluated: {greedy:4.0f}")

Compare the three lines of numbers with each other, and each line's first number with its last.

The network does learn: every run reaches episode lengths far beyond random play, with no bins and no features designed by us. But the learning is **erratic**. The curves do not climb steadily. They jump, and a level that was reached can be lost again within a few episodes. The three runs, which differ only in their random numbers, end in different places, and the policy you are left with when training stops is a matter of luck.

In [ ]:
network = runs[0][0]
video_env = gym.make("CartPole-v1", render_mode="rgb_array")
state, info = video_env.reset(seed=1)
frames = [video_env.render()]
terminated = truncated = False
while not (terminated or truncated):
    with torch.no_grad():
        action = int(network(torch.as_tensor(state)).argmax())
    state, reward, terminated, truncated, info = video_env.step(action)
    frames.append(video_env.render())

print(f"the final policy of seed 0 kept the pole up for {len(frames) - 1} steps")
show_video(frames[::2], fps=25)

## 6. Why it is unstable

The tabular algorithm was proven to converge (Notebook 05). Replacing the table by a network removes that guarantee, for two reasons that both follow from **sharing parameters**.

### Cause one: the network forgets
Consecutive steps of an episode are almost identical: the pole leans a little further, the cart moves a little on. For hundreds of updates in a row the network sees states from one small region and adjusts its shared weights to suit them, at the expense of everything else. When the cart later drifts to the other side of the track, what the network knew about that side has been overwritten. The supervised learning of other courses avoids this by **shuffling** the training data. Our agent learns from each step in the order it happens and then throws it away.

The next cell shows the mechanism on a small scale. We take a trained network, pick a "probe" state, and then update the network once on a **completely different** state.

In [ ]:
import copy

network = copy.deepcopy(runs[0][0])
optimiser = torch.optim.Adam(network.parameters(), lr=1e-3)

probe = torch.tensor([0.0, 0.0, 0.05, 0.0])            # pole leaning slightly to the right
elsewhere = torch.tensor([1.5, 1.0, -0.10, -0.5])      # a quite different situation

with torch.no_grad():
    before = network(probe).clone()

loss = (30.0 - network(elsewhere)[0]) ** 2             # one update, for the other state only
optimiser.zero_grad()
loss.backward()
optimiser.step()

with torch.no_grad():
    after = network(probe)

print("values at the probe state before:", before.numpy().round(2))
print("values at the probe state after: ", after.numpy().round(2))

The probe state was never mentioned in the update, yet its values moved. In a table they would not have changed at all. One such nudge is small. Thousands in a row, all pushing in the direction of whatever the agent happens to be doing at the moment, are not.

### Cause two: the target moves
In supervised learning the labels are fixed. Here the "label" is $R + \gamma \max_a \hat{q}(S', a; \mathbf{w})$, computed by the network being trained. Every update changes $\mathbf{w}$ and therefore changes the targets of the next updates. And because $S$ and $S'$ are neighbouring states, raising $\hat{q}(S)$ tends to raise $\hat{q}(S')$ too, which raises the target for $S$ again. The network is chasing something that runs away as it approaches, and errors can feed on themselves.

> 🧠 Sutton and Barto call the combination of **function approximation**, **bootstrapping** and **off-policy learning** the *deadly triad*: any two are safe together, and all three can make learning diverge. Q-learning with a neural network has all three.

> ⚠️ **Common pitfalls**
>
> - **Letting gradients flow through the target.** The target must be computed inside `torch.no_grad()` or with `.detach()`.
> - **Bootstrapping from a terminal state.** As with tables: when `terminated` is true the target is the reward alone.
> - **Concluding from one run.** One of the three runs above may look fine. Look at all of them before deciding that a method works.
> - **Stopping training at a lucky moment.** The agent that was good at episode 150 may be poor at episode 160.

## 7. What is needed

Each cause suggests its own cure:

| Problem | Idea | Name |
|---|---|---|
| Consecutive samples are alike, and old experience is forgotten | Keep past steps in a memory and train on **random samples** from it | experience replay |
| The target moves with every update | Compute targets with a **separate copy** of the network that is updated only slowly | target network |

These two additions, published by DeepMind in 2013 and 2015, turned the fragile learner of this notebook into an agent that learned to play dozens of Atari video games from the screen image. It is called the **Deep Q-Network**, and we build it next.

---
## ✏️ Exercises

The exercises use `env`, `make_binner`, `binned_q_learning`, `evaluate`, `make_network`, `naive_q_learning` and `smooth` from above.

### Exercise 1 (Counting rows)
Write a function `rows(bins, variables)` that returns the number of rows of a discretised table. Print it for 10 bins and 2, 4, 8, 12 and 100 variables. For which of them could the table be stored on a computer with 16 GB of memory, if each row takes 16 bytes?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def rows(bins, variables):
    return bins ** variables

limit = 16e9 / 16
for variables in (2, 4, 8, 12, 100):
    n = rows(10, variables)
    print(f"{variables:>3} variables: {n:.3e} rows   fits in memory: {n <= limit}")
```

*Eight variables still fit, in 1.6 GB. Twelve would need 16,000 GB, and that with only ten bins each. The 84 × 84 picture of Notebook 11, with 7,056 variables, is beyond any computer that will ever exist.*
</details>

### Exercise 2 (The middle ground)
Section 2 tried 3, 10 and 20 bins. Train the discretised agent with 6 bins per variable, for the seeds 0, 1 and 2, and evaluate each. Compare with the three results printed in Section 2. What do you conclude about choosing the number of bins?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for seed in range(3):
    Q, to_bins = binned_q_learning(env, 6, seed=seed)
    score = evaluate(env, lambda state: int(Q[to_bins(state)].argmax()))
    print(f"6 bins, seed {seed}: average episode length {score:.0f}")
```

*Six bins do better here than any of the three values tried in the notebook. Nothing told us that in advance: the number of bins is a setting that has to be found by trying, and the best value would be different for another problem. The results also vary from seed to seed, so a fair comparison of two settings needs several runs of each.*
</details>

### Exercise 3 (The learning rate)
Run `naive_q_learning` with a learning rate of `1e-4` (ten times smaller than in the notebook) for seeds 0 and 1, and print the best 10-episode average and the average of the last 10 training episodes for each. What goes wrong? Hint: think about how fast the exploration is being reduced.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for seed in range(2):
    network, lengths = naive_q_learning(env, learning_rate=1e-4, seed=seed)
    print(f"learning rate 0.0001, seed {seed}: best 10-episode average {smooth(lengths).max():4.0f},  "
          f"last 10 episodes {lengths[-10:].mean():4.0f}")
```

*With such small steps the network has learned very little by the time exploration has faded. From then on the agent acts greedily on values that are still almost arbitrary, and the pole falls quickly. The learning rate, the exploration schedule and the length of training have to fit together: changing one of them alone can turn a working setup into a failing one.*
</details>

### Exercise 4 (A smaller network)
Train the naive agent with only 8 hidden units per layer (`hidden=8`) for seeds 0 and 1, and compare with the 64-unit runs of the notebook: the number of parameters, the best 10-episode average and the last 10 episodes. Is a larger network always the more stable one?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
small = make_network(8)
print("parameters with 8 hidden units: ", sum(p.numel() for p in small.parameters()))
print("parameters with 64 hidden units:", sum(p.numel() for p in make_network(64).parameters()))

for seed in range(2):
    network, lengths = naive_q_learning(env, hidden=8, seed=seed)
    print(f" 8 hidden units, seed {seed}: best 10-episode average {smooth(lengths).max():4.0f},  "
          f"last 10 episodes {lengths[-10:].mean():4.0f}")
for seed in range(2):
    lengths = runs[seed][1]
    print(f"64 hidden units, seed {seed}: best 10-episode average {smooth(lengths).max():4.0f},  "
          f"last 10 episodes {lengths[-10:].mean():4.0f}")
```

*Compare the numbers you get. A small network has less capacity to represent the values, and it also shares its few parameters even more heavily between states, so an update in one place disturbs more of the rest. There is no simple rule that bigger or smaller is safer. The instability comes from the training procedure, and that is what the next notebook changes.*
</details>

### Exercise 5 (A straight-line value function (a little harder))
How complicated does the value function have to be? Replace the network by a single linear layer, `nn.Linear(4, 2)`, with no hidden layer: each action value is a weighted sum of the four state variables. Write `linear_q_learning` by adapting `naive_q_learning`, train it for 20,000 steps with seeds 0 and 1, evaluate the final policies, and print the learned weights of one of them. Then evaluate the hand-written rule "push right if pole angle + turning speed is positive", which is also linear. What does the comparison tell you?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def linear_q_learning(env, steps=20000, learning_rate=1e-3, gamma=0.99, seed=0):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = nn.Linear(4, 2)
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
    state, info = env.reset(seed=seed)
    for step in range(steps):
        epsilon = max(0.01, 1 - step / (0.5 * steps))
        s = torch.as_tensor(state)
        with torch.no_grad():
            action = int(rng.integers(2)) if rng.random() < epsilon else int(model(s).argmax())
        state, reward, terminated, truncated, info = env.step(action)
        with torch.no_grad():
            target = reward + gamma * (0.0 if terminated else model(torch.as_tensor(state)).max())
        loss = (target - model(s)[action]) ** 2
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        if terminated or truncated:
            state, info = env.reset()
    return model


for seed in range(2):
    model = linear_q_learning(env, seed=seed)
    score = evaluate(env, lambda state: int(model(torch.as_tensor(state)).argmax()))
    print(f"linear model, seed {seed}: average episode length {score:.0f}")

difference = (model.weight[1] - model.weight[0]).detach().numpy()
print("weight of each variable in 'push right' minus 'push left':", difference.round(2))
print("variables: cart position, cart speed, pole angle, pole turning speed")

by_hand = evaluate(env, lambda state: int(state[2] + state[3] > 0))
print(f"\na rule written by hand, 'push right if angle + turning speed > 0': {by_hand:.0f}")
```

*The linear agent fails: its pole falls almost at once. Yet the last line shows that a **linear rule** balances the pole well. The rule is "push towards the side the pole is falling to", and it needs only a sum of two state variables. So a linear model is able to express a good **policy**. What it cannot express is the **value function** we asked it to learn: the true action values are far from linear in the state (they are high in the middle and drop off steeply towards every edge). A greedy policy read off badly fitting values is a bad policy. Module D takes the hint and learns the policy directly, without going through values.*
</details>

## 🔑 Key takeaways

- When states are real numbers, **no state repeats**, so a table has nothing to accumulate. The agent must **generalise**.
- **Discretisation** makes a table usable again, but the number of rows grows as $b^d$: the **curse of dimensionality**.
- **Function approximation** computes action values with a parametrised function. A neural network takes the state and returns **one value per action**.
- The Q-learning update becomes a **loss**: the squared difference between $\hat{q}(S, A)$ and a target that is held fixed.
- Sharing parameters gives generalisation and takes away the convergence guarantee: the network **forgets**, and its **targets move**.
- Function approximation, bootstrapping and off-policy learning together form the **deadly triad**.

---
**Next:** *Notebook 09: Deep Q-Networks*, where two additions turn this erratic learner into a reliable one.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*